In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv(
    "data/processed/content_candidates.csv"
)

ratings = pd.read_csv("data/ratings.csv")

print("=== Data Shape ===")
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

=== Data Shape ===
Movies: (9708, 26)
Ratings: (100836, 4)


In [2]:
selected_genres = [
    "Action",
    "Sci-Fi",
    "Thriller"
]

rating_history = {
    2571: 5.0,
    1: 4.5,
    2: 2.0
}

available_genres = [
    "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western"
]

print("=== User Information ===")
print("Selected genres:", selected_genres)
print("Rating history:", rating_history)

=== User Information ===
Selected genres: ['Action', 'Sci-Fi', 'Thriller']
Rating history: {2571: 5.0, 1: 4.5, 2: 2.0}


In [4]:
def build_user_profile(
    selected_genres,
    rating_history,
    movies,
    available_genres,
    baseline=3.5
):
    # 1. Create the initial user profile
    profile = pd.Series(
        0.0,
        index=available_genres
    )

    profile[selected_genres] = 1.0

    # 2. Apply rating records one at a time
    for movie_id, rating in rating_history.items():

        profile = update_user_profile(
            user_profile=profile,
            movie_id=movie_id,
            rating=rating,
            movies=movies,
            available_genres=available_genres,
            baseline=baseline
        )

    return profile

In [5]:
def update_user_profile(
    user_profile,
    movie_id,
    rating,
    movies,
    available_genres,
    baseline=3.5
):
    # 1. Validate the rating range
    if rating < 0.5 or rating > 5.0:
        raise ValueError("Rating must be between 0.5 and 5.0")

    if rating * 2 != round(rating * 2):
        raise ValueError("Rating must be in 0.5 increments")

    # 2. Search for the movie
    movie = movies[movies["movieId"] == movie_id]

    if movie.empty:
        raise ValueError("Movie not found")

    # 3. Retrieve the genre vector
    genre_vector = movie[available_genres].iloc[0]

    # 4. Count the movie's genres
    genre_count = genre_vector.sum()

    if genre_count == 0:
        return user_profile.copy()

    # 5. Calculate preference change based on the rating
    preference_weight = rating - baseline
    genre_weight = preference_weight / genre_count

    # 6. Update the user profile
    new_profile = user_profile.copy()

    new_profile += genre_vector * genre_weight

    return new_profile

In [6]:
user_profile = build_user_profile(
    selected_genres=selected_genres,
    rating_history=rating_history,
    movies=movies,
    available_genres=available_genres
)

print("=== Hybrid User Profile ===")
print(user_profile)

=== Hybrid User Profile ===
Action         1.5
Adventure     -0.3
Animation      0.2
Children      -0.3
Comedy         0.2
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy       -0.3
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.5
Thriller       1.5
War            0.0
Western        0.0
dtype: float64


In [8]:
def recommend_movies(
    user_profile,
    movies,
    available_genres,
    watched_movie_ids=None,
    alpha=0.7,
    top_n=10
):
    # 1. Get the list of rated movies
    if watched_movie_ids is None:
        watched_movie_ids = []

    # 2. Prepare the movie genre matrix
    movie_vectors = movies[available_genres].to_numpy()

    # 3. Prepare the user vector
    user_vector = user_profile[
        available_genres
    ].to_numpy().reshape(1, -1)

    # 4. Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    # 5. Store results
    results = movies.copy()

    results["genre_similarity"] = similarity_scores

    # 6. Calculate final recommendation scores
    results["recommendation_score"] = (
        alpha * results["genre_similarity"]
        + (1 - alpha) * results["normalized_rating"]
    )

    # 7. Exclude watched movies
    results = results[
        ~results["movieId"].isin(watched_movie_ids)
    ]

    # 8. Exclude movies with zero genre similarity
    results = results[
        results["genre_similarity"] > 0
    ]

    # 9. Sort by recommendation score
    results = results.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return results.head(top_n)

In [9]:
content_results = recommend_movies(
    user_profile=user_profile,
    movies=movies,
    available_genres=available_genres,
    watched_movie_ids=list(rating_history.keys()),
    alpha=0.7,
    top_n=len(movies)
)

print("=== Content-Based Results ===")

print("Number of movies:", len(content_results))

print(
    content_results[
        [
            "movieId",
            "title",
            "genre_similarity",
            "recommendation_score"
        ]
    ].head(10).to_string(index=False)
)

=== Content-Based Results ===
Number of movies: 6119
 movieId                             title  genre_similarity  recommendation_score
     541               Blade Runner (1982)          0.975041              0.934258
    1240            Terminator, The (1984)          0.975041              0.908036
   44191             V for Vendetta (2006)          0.975041              0.905407
    5903                Equilibrium (2002)          0.975041              0.899202
    2600                   eXistenZ (1999)          0.975041              0.891447
    5181                  Hangar 18 (1980)          0.975041              0.879886
    4553                  They Live (1988)          0.975041              0.871217
  122920 Captain America: Civil War (2016)          0.975041              0.867369
    3527                   Predator (1987)          0.975041              0.866252
  102125                 Iron Man 3 (2013)          0.975041              0.863079


In [10]:
user_item_matrix = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

print("=== User-Item Matrix ===")
print(user_item_matrix.shape)

=== User-Item Matrix ===
(610, 9724)


In [12]:
def calculate_all_item_similarities(
    movie_id,
    user_item_matrix,
    min_common_users=10,
    shrinkage=20
):
    # 1. Get ratings for the target movie
    target_ratings = user_item_matrix[movie_id]

    # 2. Identify users who rated the target movie
    target_users = target_ratings.dropna().index

    # 3. Get all movie ratings from those users
    subset = user_item_matrix.loc[target_users]

    # 4. Count common raters
    common_counts = subset.notna().sum(axis=0)

    # 5. Keep movies with sufficient common raters
    valid_movies = common_counts[
        common_counts >= min_common_users
    ].index

    subset_valid = subset[valid_movies]

    # 6. Check rating variance among common raters for both movies
    target_common = pd.DataFrame(
        {
            col: target_ratings.loc[target_users]
            for col in valid_movies
        }
    ).where(subset_valid.notna())

    valid_variance = (
        (subset_valid.std(axis=0) > 0)
        & (target_common.std(axis=0) > 0)
    )

    subset_valid = subset_valid.loc[
        :, valid_variance
    ]

    # 7. Calculate Pearson correlations
    similarities = subset_valid.corrwith(
        target_ratings.loc[target_users],
        axis=0
    )

    # 8. Apply shrinkage
    adjusted = (
        similarities
        * common_counts.loc[similarities.index]
        / (
            common_counts.loc[similarities.index]
            + shrinkage
        )
    )

    # 9. Restore the full movie index
    results = pd.Series(
        np.nan,
        index=user_item_matrix.columns,
        dtype=float
    )

    results.loc[adjusted.index] = adjusted

    # 10. Exclude the target movie itself
    results.loc[movie_id] = np.nan

    return results

In [13]:
similarity_data = {}

for movie_id in rating_history:
    similarity_data[movie_id] = (
        calculate_all_item_similarities(
            movie_id=movie_id,
            user_item_matrix=user_item_matrix,
            min_common_users=10,
            shrinkage=20
        )
    )

similarity_df = pd.DataFrame(similarity_data)

# Use only positive similarities
positive_similarities = (
    similarity_df.clip(lower=0).fillna(0)
)

# Calculate preference deviations from user ratings
preferences = pd.Series({
    movie_id: rating - 3.5
    for movie_id, rating in rating_history.items()
})

# Calculate CF scores
numerator = positive_similarities.mul(
    preferences,
    axis=1
).sum(axis=1)

similarity_strength = positive_similarities.sum(axis=1)

positive_support = (
    positive_similarities > 0
).sum(axis=1)

lambda_cf = 0.5

cf_scores = numerator / (
    similarity_strength + lambda_cf
)

print("=== CF Score Statistics ===")
print(cf_scores.describe())

print(
    "Movies with at least 2 positive similarities:",
    (positive_support >= 2).sum()
)

=== CF Score Statistics ===
count    9724.000000
mean        0.024069
std         0.112140
min        -0.544151
25%         0.000000
50%         0.000000
75%         0.000000
max         0.661368
dtype: float64
Movies with at least 2 positive similarities: 1307


In [14]:
cf_results = pd.DataFrame({
    "movieId": cf_scores.index,
    "cf_score": cf_scores.values,
    "positive_support": positive_support.values,
    "similarity_strength": similarity_strength.values
})

# Identify sufficiently supported CF scores
cf_results["cf_available"] = (
    cf_results["positive_support"] >= 2
)

# Do not use CF scores with insufficient support
cf_results.loc[
    ~cf_results["cf_available"],
    "cf_score"
] = np.nan

print("=== CF Results ===")
print(cf_results.head(10))

print(
    "\nMovies with available CF scores:",
    cf_results["cf_available"].sum()
)

=== CF Results ===
   movieId  cf_score  positive_support  similarity_strength  cf_available
0        1 -0.200996                 2             0.391988          True
1        2  0.520428                 2             0.396185          True
2        3  0.126014                 3             0.672729          True
3        4       NaN                 0             0.000000         False
4        5 -0.241879                 3             0.493220          True
5        6  0.271653                 3             0.399521          True
6        7 -0.210584                 3             0.333368          True
7        8       NaN                 0             0.000000         False
8        9       NaN                 1             0.035967         False
9       10  0.282154                 2             0.146020          True

Movies with available CF scores: 1307


In [15]:
cf_results["cf_normalized"] = (
    cf_results["cf_score"] + 3.0
) / 4.5

print("=== Normalized CF Results ===")

print(
    cf_results[
        cf_results["cf_available"]
    ][
        [
            "movieId",
            "cf_score",
            "cf_normalized",
            "positive_support"
        ]
    ].head(10).to_string(index=False)
)

=== Normalized CF Results ===
 movieId  cf_score  cf_normalized  positive_support
       1 -0.200996       0.622001                 2
       2  0.520428       0.782317                 2
       3  0.126014       0.694670                 3
       5 -0.241879       0.612916                 3
       6  0.271653       0.727034                 3
       7 -0.210584       0.619870                 3
      10  0.282154       0.729368                 2
      11 -0.188429       0.624794                 3
      17  0.101032       0.689118                 3
      18  0.006180       0.668040                 3


In [16]:
hybrid_data = movies.merge(
    content_results[
        ["movieId", "genre_similarity", "recommendation_score"]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

hybrid_data = hybrid_data.merge(
    cf_results[
        [
            "movieId",
            "cf_score",
            "cf_normalized",
            "positive_support",
            "cf_available"
        ]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

print("=== Hybrid Data ===")

print(
    hybrid_data[
        [
            "movieId",
            "title",
            "recommendation_score",
            "cf_score",
            "cf_normalized",
            "positive_support"
        ]
    ].head(10).to_string(index=False)
)

=== Hybrid Data ===
 movieId                              title  recommendation_score  cf_score  cf_normalized  positive_support
       1                   Toy Story (1995)                   NaN -0.200996       0.622001               2.0
       2                     Jumanji (1995)                   NaN  0.520428       0.782317               2.0
       3            Grumpier Old Men (1995)              0.182772  0.126014       0.694670               3.0
       4           Waiting to Exhale (1995)              0.138367       NaN            NaN               0.0
       5 Father of the Bride Part II (1995)              0.176544 -0.241879       0.612916               3.0
       6                        Heat (1995)              0.685777  0.271653       0.727034               3.0
       7                     Sabrina (1995)              0.173803 -0.210584       0.619870               3.0
       8                Tom and Huck (1995)                   NaN       NaN            NaN               0.0

In [17]:
print("=== Hybrid Data Validation ===")

print("Total movies:", len(hybrid_data))

print(
    "Movies with Content-Based scores:",
    hybrid_data["recommendation_score"].notna().sum()
)

print(
    "Movies with CF scores:",
    hybrid_data["cf_score"].notna().sum()
)

print(
    "Movies with both scores:",
    (
        hybrid_data["recommendation_score"].notna()
        & hybrid_data["cf_score"].notna()
    ).sum()
)

print(
    "Duplicate movie IDs:",
    hybrid_data["movieId"].duplicated().sum()
)

=== Hybrid Data Validation ===
Total movies: 9708
Movies with Content-Based scores: 6119
Movies with CF scores: 1307
Movies with both scores: 955
Duplicate movie IDs: 0


In [18]:
# 1. Select movies with content-based scores
hybrid_candidates = hybrid_data[
    hybrid_data["recommendation_score"].notna()
].copy()

# 2. Convert CF scores into a recommendation adjustment signal
# Missing CF scores have no effect
hybrid_candidates["cf_signal"] = (
    hybrid_candidates["cf_score"].fillna(0.0) / 1.5
)

# 3. Calculate hybrid scores
beta = 0.2

hybrid_candidates["hybrid_score"] = (
    hybrid_candidates["recommendation_score"]
    + beta * hybrid_candidates["cf_signal"]
).clip(0, 1)

print("=== Hybrid Score Examples ===")

print(
    hybrid_candidates[
        [
            "title",
            "recommendation_score",
            "cf_score",
            "cf_signal",
            "hybrid_score"
        ]
    ].head(10).to_string(index=False)
)

=== Hybrid Score Examples ===
                             title  recommendation_score  cf_score  cf_signal  hybrid_score
           Grumpier Old Men (1995)              0.182772  0.126014   0.084009      0.199573
          Waiting to Exhale (1995)              0.138367       NaN   0.000000      0.138367
Father of the Bride Part II (1995)              0.176544 -0.241879  -0.161253      0.144293
                       Heat (1995)              0.685777  0.271653   0.181102      0.721998
                    Sabrina (1995)              0.173803 -0.210584  -0.140390      0.145725
               Sudden Death (1995)              0.535642       NaN   0.000000      0.535642
                  GoldenEye (1995)              0.582867  0.282154   0.188103      0.620488
    American President, The (1995)              0.225203 -0.188429  -0.125619      0.200079
Dracula: Dead and Loving It (1995)              0.112027       NaN   0.000000      0.112027
           Cutthroat Island (1995)              0.

In [19]:
watched_movie_ids = list(rating_history.keys())

hybrid_candidates = hybrid_candidates[
    ~hybrid_candidates["movieId"].isin(watched_movie_ids)
].copy()

print("=== Watched Movie Validation ===")

print(
    "Watched movie IDs:",
    watched_movie_ids
)

print(
    "Watched movies in candidates:",
    hybrid_candidates["movieId"]
    .isin(watched_movie_ids)
    .sum()
)

=== Watched Movie Validation ===
Watched movie IDs: [2571, 1, 2]
Watched movies in candidates: 0


In [20]:
hybrid_top10 = hybrid_candidates.sort_values(
    by="hybrid_score",
    ascending=False
).head(10)

print("=== Hybrid Recommendation Top 10 ===")

print(
    hybrid_top10[
        [
            "title",
            "genres",
            "recommendation_score",
            "cf_score",
            "hybrid_score"
        ]
    ].to_string(index=False)
)

=== Hybrid Recommendation Top 10 ===
                            title                      genres  recommendation_score  cf_score  hybrid_score
            V for Vendetta (2006) Action|Sci-Fi|Thriller|IMAX              0.905407  0.389393      0.957326
              Blade Runner (1982)      Action|Sci-Fi|Thriller              0.934258  0.119105      0.950138
               Equilibrium (2002)      Action|Sci-Fi|Thriller              0.899202  0.232802      0.930242
           Terminator, The (1984)      Action|Sci-Fi|Thriller              0.908036  0.122384      0.924354
               Island, The (2005)      Action|Sci-Fi|Thriller              0.841034  0.537161      0.912655
                Iron Man 3 (2013) Action|Sci-Fi|Thriller|IMAX              0.863079  0.356028      0.910550
              Transformers (2007) Action|Sci-Fi|Thriller|IMAX              0.839247  0.515729      0.908011
                  eXistenZ (1999)      Action|Sci-Fi|Thriller              0.891447       NaN      

In [21]:
content_top10 = content_results.sort_values(
    by="recommendation_score",
    ascending=False
).head(10)

comparison = pd.DataFrame({
    "Content-Based": content_top10["title"].values,
    "Hybrid": hybrid_top10["title"].values
})

comparison.index = range(1, 11)

print("=== Recommendation Ranking Comparison ===")
print(comparison.to_string())

=== Recommendation Ranking Comparison ===
                        Content-Based                             Hybrid
1                 Blade Runner (1982)              V for Vendetta (2006)
2              Terminator, The (1984)                Blade Runner (1982)
3               V for Vendetta (2006)                 Equilibrium (2002)
4                  Equilibrium (2002)             Terminator, The (1984)
5                     eXistenZ (1999)                 Island, The (2005)
6                    Hangar 18 (1980)                  Iron Man 3 (2013)
7                    They Live (1988)                Transformers (2007)
8   Captain America: Civil War (2016)                    eXistenZ (1999)
9                     Predator (1987)  Captain America: Civil War (2016)
10                  Iron Man 3 (2013)                    Predator (1987)


In [22]:
content_ids = set(content_top10["movieId"])
hybrid_ids = set(hybrid_top10["movieId"])

overlap = len(content_ids & hybrid_ids)

print("=== Recommendation Overlap ===")
print("Common recommended movies:", overlap)
print("Overlap@10:", overlap / 10)

=== Recommendation Overlap ===
Common recommended movies: 8
Overlap@10: 0.8


In [23]:
print("=== Hybrid Recommendation Validation ===")

print("Number of candidates:", len(hybrid_candidates))

print(
    "Missing hybrid scores:",
    hybrid_candidates["hybrid_score"].isna().sum()
)

print(
    "Minimum hybrid score:",
    hybrid_candidates["hybrid_score"].min()
)

print(
    "Maximum hybrid score:",
    hybrid_candidates["hybrid_score"].max()
)

print(
    "Number of recommendations:",
    len(hybrid_top10)
)

print(
    "Watched movies recommended:",
    hybrid_top10["movieId"]
    .isin(rating_history.keys())
    .sum()
)

=== Hybrid Recommendation Validation ===
Number of candidates: 6119
Missing hybrid scores: 0
Minimum hybrid score: 0.07045998157360282
Maximum hybrid score: 0.9573262643713265
Number of recommendations: 10
Watched movies recommended: 0
